In [1]:
import os
import json
import requests
import getpass
from base64 import b64encode
from datetime import datetime, timezone
from netCDF4 import Dataset
from urllib.parse import urlparse

In [2]:
# URL for the CEDA Token API service
TOKEN_URL = "https://services-beta.ceda.ac.uk/api/token/create/"
# Location on the filesystem to store a cached download token
TOKEN_CACHE = os.path.expanduser(os.path.join("~", ".cedatoken"))

In [3]:
def load_cached_token():
    try:
        with open(TOKEN_CACHE, "r") as cache_file:
            data = json.loads(cache_file.read())
            token = data.get("access_token")
            expires = datetime.strptime(data.get("expires"), "%Y-%m-%dT%H:%M:%S.%f%z")
            return token, expires
    except FileNotFoundError:
        return None, None

In [4]:
def get_token():
    token, expires = load_cached_token()
    now = datetime.now(timezone.utc)
    if not token or expires < now:
        username = input("Please provide your CEDA username: ")
        password = getpass.getpass(prompt="CEDA user password: ")
        credentials = b64encode(f"{username}:{password}".encode("utf-8")).decode("ascii")
        headers = {"Authorization": f"Basic {credentials}"}
        response = requests.request("POST", TOKEN_URL, headers=headers)
        if response.status_code == 200:
            response_data = json.loads(response.text)
            token = response_data["access_token"]
            with open(TOKEN_CACHE, "w") as cache_file:
                cache_file.write(response.text)
        else:
            print("Failed to generate token, check your username and password.")
    else:
        print(f"Using cached token, expires at {expires}.")
    return token

In [5]:
def open_dataset(url, download_token=None):
    headers = {"Authorization": f"Bearer {download_token}"} if download_token else None
    response = requests.get(url, headers=headers, stream=True)
    if response.status_code != 200:
        print(f"Failed to fetch data. Response code: {response.status_code}")
        return None
    filename = os.path.basename(urlparse(url).path)
    return Dataset(filename, memory=response.content)

In [6]:
# Interactive Inputs
url = input("Enter the NetCDF file URL: ")
var_id = input("Enter the variable ID: ")

Enter the NetCDF file URL:  https://dap.ceda.ac.uk/badc/faam/data/2012/b749-oct-03/core_processed/core_faam_20121003_v004_r0_b749_1hz.nc
Enter the variable ID:  Test_netcdf


In [7]:
token = get_token()
if token:
    dataset = open_dataset(url, download_token=token)
    if dataset:
        print("\n[INFO] Global attributes:")
        for attr in dataset.ncattrs():
            print(f"\t{attr}: {dataset.getncattr(attr)}")
        print("\n[INFO] Variables:")
        print(dataset.variables)
        print("\n[INFO] Dimensions:")
        print(dataset.dimensions)
        if var_id in dataset.variables:
            variable = dataset.variables[var_id][:]
            units = dataset.variables[var_id].units
            print(f"\n[INFO] {var_id} - Min: {variable.min()} {units}, Max: {variable.max()} {units}")
    else:
        print("Failed to open dataset.")
else:
    print("Aborting since no token was retrieved.")

Please provide your CEDA username:  tnobrega
CEDA user password:  ········



[INFO] Global attributes:
	conventions: CF-1.0
	data_date: 20121015
	title: Data from b749 on 20121003
	TimeInterval: 10:10:55-17:34:34
	INSTITUTION: FAAM
	SOURCE: FAAM BAe-146 Aircraft Data
	REFERENCES: http://www.faam.ac.uk
	CAL_INPUT_FILES: Input files:
/home/tardis/data/fltcons/flight-cst_faam_20121003_r0_b749.txt
/home/tardis/tardis/caltext/MFDPARDESC.DAT
/home/tardis/data/rawdata/b749_raw_hddr.dat
/home/tardis/data/rawdata/b749_raw_data.dat
C_TWC           V1.01

	CAL_OUTPUT_FILES: Output files (M5 dataset):
/home/tardis/data/mfddata/b749_mfda_hddr.dat  112.5 Kbytes (   225 blocks)
/home/tardis/data/mfddata/b749_mfda_data.dat  123.9 Mbytes (253724 blocks)
/home/tardis/data/mfddata/b749_mfda_text.dat

	CAL_MODULES: Modules used:
DRS    RADAL1 GENEAS LWC    TURB   SOLS   TWC    INS1   HEIMAN NEPHL1 OZONE1
PSAP   TEMPS2 AIRSPD PRESS1 RVSM   TPRESS COMR   FWVS   GWINDS

	CAL_COMMAND: Command line:
/home/tardis/data/fltcons/flight-cst_faam_20121003_r0_b749.txt /home/tardis/dat
a/rawd